# 07 · Embeddings in PyTorch · Word Embeddings from Scratch

| | |
|---|---|
| **Author** | Chandan C |
| **GitHub** | [github.com/chandanc5525](https://github.com/chandanc5525) |
| **Course** | Agentic AI BootCamp · Module 02 – Advanced PyTorch |
| **Repository** | [AgenticAI_BootCamp-/02_advanced_pytorch](https://github.com/chandanc5525/AgenticAI_BootCamp-/tree/main/02_advanced_pytorch) |
| **Level** | Intermediate |
| **Prerequisites** | Notebooks 01-03 |

---

### What you will learn
- Why dense vectors beat one-hot, and `nn.Embedding` as a lookup table
- `nn.Embedding`, `padding_idx`, `max_norm`, `from_pretrained`, `EmbeddingBag`, sparse gradients
- Word2Vec skip-gram with negative sampling (SGNS) implemented from scratch
- GloVe-style co-occurrence factorisation
- FastText-style sub-word (hashed n-gram) embeddings for unseen words
- Inspecting embeddings: cosine neighbours, PCA, analogies
- Re-using embeddings downstream (frozen vs fine-tuned), saving and loading

> Every cell runs on a plain CPU. If a GPU (CUDA / Apple MPS) is present, the notebooks use it automatically.

## 0. Setup

In [ ]:
import math, random, re, collections, itertools, os, tempfile
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

def seed_everything(s=0):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
seed_everything()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.__version__, device)

## 1. From one-hot to dense vectors
A one-hot vector for a vocabulary of size `V` is huge, sparse, and every pair of words is equally distant. An **embedding** maps each id to a short dense vector (`d ≪ V`) that the model *learns*, so similar words end up close together.

`nn.Embedding(V, d)` is just a trainable `V × d` matrix indexed by token id. Multiplying a one-hot vector with that matrix gives the same result - the lookup is simply a much cheaper way of doing it.

In [ ]:
V, d = 10, 4
emb = nn.Embedding(V, d)
ids = torch.tensor([3, 7, 3])

lookup = emb(ids)                                       # (3, 4)  - O(1) per token
one_hot = F.one_hot(ids, num_classes=V).float()          # (3, 10) - wasteful
matmul = one_hot @ emb.weight                            # same result via matmul
print("lookup == one_hot @ W :", torch.allclose(lookup, matmul))
print("weight matrix shape   :", tuple(emb.weight.shape), "| trainable:", emb.weight.requires_grad)

# One-hot: every pair is equally (dis)similar. Learned vectors can encode similarity.
print("one-hot cosine(3,7) =", F.cosine_similarity(one_hot[0], one_hot[1], dim=0).item())

## 2. The `nn.Embedding` toolbox

In [ ]:
# padding_idx: that row is a constant zero vector and receives no gradient
e = nn.Embedding(5, 3, padding_idx=0)
print("pad row:", e.weight[0].tolist())
e(torch.tensor([[0, 2, 4]])).sum().backward()
print("grad of pad row:", e.weight.grad[0].tolist(), "| grad of row 2:", e.weight.grad[2].tolist())

# max_norm: renormalise looked-up vectors so their L2 norm never exceeds max_norm
e2 = nn.Embedding(5, 3, max_norm=1.0); _ = e2(torch.arange(5)); print("max row norm:", e2.weight.norm(dim=1).max().item())

# from_pretrained: load vectors (GloVe, fastText, your own) and freeze or fine-tune them
vectors = torch.randn(5, 3)
frozen = nn.Embedding.from_pretrained(vectors, freeze=True)
tuned  = nn.Embedding.from_pretrained(vectors, freeze=False, padding_idx=0)
print("frozen requires_grad:", frozen.weight.requires_grad, "| tuned:", tuned.weight.requires_grad)

# EmbeddingBag: embed AND pool variable-length bags in a single fused op (no padding needed)
bag = nn.EmbeddingBag(10, 4, mode="mean")
flat = torch.tensor([1, 2, 4, 5, 4, 3, 2, 9]); offsets = torch.tensor([0, 4])  # two bags of 4 tokens
print("EmbeddingBag out:", bag(flat, offsets).shape)

# sparse=True: only the rows that were used get gradient updates (huge vocabularies)
big = nn.Embedding(100_000, 32, sparse=True)
opt = torch.optim.SparseAdam(big.parameters(), lr=1e-3)
big(torch.tensor([5, 42])).sum().backward(); opt.step()
print("sparse grad layout:", big.weight.grad.layout)

## 3. A small corpus with real structure
To learn embeddings offline we generate a synthetic corpus from templates. Words in the same group (animals, foods, countries, capitals, male/female roles) appear in similar contexts, so a good embedding should place them together, and relations such as *country → capital* and *male → female* should become consistent **directions** in the space.

In [ ]:
animals = ["cat", "dog", "horse", "cow", "sheep", "goat"]
foods = ["bread", "apple", "rice", "cheese", "fish", "soup"]
pairs_cc = [("france", "paris"), ("italy", "rome"), ("japan", "tokyo"), ("spain", "madrid"), ("germany", "berlin"), ("india", "delhi")]
pairs_g = [("king", "queen"), ("man", "woman"), ("boy", "girl"), ("uncle", "aunt"), ("prince", "princess")]
places = ["house", "garden", "market", "school"]

def make_corpus(n=12000, seed=0):
    rnd = random.Random(seed); out = []
    for _ in range(n):
        k = rnd.random()
        if k < 0.25:
            a, f = rnd.choice(animals), rnd.choice(foods)
            out.append(rnd.choice([f"the {a} eats the {f} in the {rnd.choice(places)}",
                                   f"a {a} is a small animal that likes {f}",
                                   f"the farm has a {a} and a {rnd.choice(animals)}"]))
        elif k < 0.55:
            c, cap = rnd.choice(pairs_cc)
            out.append(rnd.choice([f"{cap} is the capital of {c}", f"{cap} is a big city in {c}",
                                   f"people in {c} live in {cap} and eat {rnd.choice(foods)}",
                                   f"the capital city {cap} is in the country {c}"]))
        else:
            m, f = rnd.choice(pairs_g)
            if rnd.random() < 0.5:
                out.append(rnd.choice([f"the {m} said he is happy", f"the {m} lives in the {rnd.choice(places)} and he works",
                                       f"he is the {m} of the {rnd.choice(places)}"]))
            else:
                out.append(rnd.choice([f"the {f} said she is happy", f"the {f} lives in the {rnd.choice(places)} and she works",
                                       f"she is the {f} of the {rnd.choice(places)}"]))
    return out

corpus = make_corpus()
tokens = [s.split() for s in corpus]
counts = collections.Counter(w for s in tokens for w in s)
itos = [w for w, _ in counts.most_common()]; stoi = {w: i for i, w in enumerate(itos)}
V = len(itos)
print(f"{len(corpus)} sentences | vocab {V} | sample: {corpus[:3]}")

## 4. Word2Vec - skip-gram with negative sampling (SGNS)
**Idea:** given a centre word, predict the words around it. A full softmax over `V` words is expensive, so we turn it into binary classification: *is this (centre, context) pair real or a randomly drawn negative?*

$$\mathcal L = -\log\sigma(u_o^\top v_c) - \sum_{k=1}^{K}\log\sigma(-u_{n_k}^\top v_c),\qquad n_k\sim P_n(w)\propto f(w)^{3/4}$$

Two tables are learned: `in_embed` (centre vectors `v`) and `out_embed` (context vectors `u`). After training we use `in_embed`, or the sum / average of both tables (usually a bit better, and what we do here).

In [ ]:
def build_pairs(tokens, window=3):
    centres, contexts = [], []
    for sent in tokens:
        ids = [stoi[w] for w in sent]
        for i, c in enumerate(ids):
            for j in range(max(0, i - window), min(len(ids), i + window + 1)):
                if j != i: centres.append(c); contexts.append(ids[j])
    return torch.tensor(centres), torch.tensor(contexts)

centres, contexts = build_pairs(tokens)
print("training pairs:", len(centres))

# noise distribution: unigram^(3/4)
freq = torch.tensor([counts[w] for w in itos], dtype=torch.float)
noise = (freq ** 0.75) / (freq ** 0.75).sum()

class SkipGramNS(nn.Module):
    def __init__(self, V, d):
        super().__init__()
        self.in_embed = nn.Embedding(V, d); self.out_embed = nn.Embedding(V, d)
        nn.init.uniform_(self.in_embed.weight, -0.5 / d, 0.5 / d); nn.init.zeros_(self.out_embed.weight)
    def forward(self, centre, context, negatives):
        v = self.in_embed(centre)                       # (B, d)
        u_pos = self.out_embed(context)                 # (B, d)
        u_neg = self.out_embed(negatives)               # (B, K, d)
        pos = F.logsigmoid((v * u_pos).sum(-1))
        neg = F.logsigmoid(-(u_neg @ v.unsqueeze(-1)).squeeze(-1)).sum(-1)
        return -(pos + neg).mean()

def train_sgns(d=32, K=5, epochs=8, bs=1024, lr=5e-3):
    model = SkipGramNS(V, d).to(device); opt = torch.optim.Adam(model.parameters(), lr=lr)
    n = len(centres)
    for ep in range(epochs):
        perm = torch.randperm(n); total = 0
        for i in range(0, n, bs):
            idx = perm[i:i + bs]
            c, o = centres[idx].to(device), contexts[idx].to(device)
            neg = torch.multinomial(noise, len(idx) * K, replacement=True).view(len(idx), K).to(device)
            loss = model(c, o, neg); opt.zero_grad(); loss.backward(); opt.step(); total += loss.item() * len(idx)
        print(f"epoch {ep} | loss {total / n:.4f}")
    return model

seed_everything()
sgns = train_sgns()
W_in = F.normalize(sgns.in_embed.weight.detach().cpu(), dim=-1)
W = F.normalize((sgns.in_embed.weight + sgns.out_embed.weight).detach().cpu(), dim=-1)   # in + out tables, unit vectors: dot = cosine

## 5. Inspecting the learned space

In [ ]:
def neighbours(word, E=None, k=5):
    E = W if E is None else E
    sims = E @ E[stoi[word]]
    return [(itos[i], round(s, 3)) for s, i in zip(*[t.tolist() for t in sims.topk(k + 1)])][1:]

for w in ["cat", "bread", "paris", "france", "queen", "she"]:
    print(f"{w:7s} -> {neighbours(w)}")

In [ ]:
# 2-D projection with PCA (low-rank SVD in PyTorch)
def pca2(E):
    X = E - E.mean(0, keepdim=True)
    U, S, Vh = torch.linalg.svd(X, full_matrices=False)
    return (X @ Vh[:2].T).numpy()

P = pca2(W)
groups = {"animal": animals, "food": foods, "country": [c for c, _ in pairs_cc], "capital": [c for _, c in pairs_cc],
          "male": [m for m, _ in pairs_g], "female": [f for _, f in pairs_g]}
colors = dict(zip(groups, ["tab:brown", "tab:orange", "tab:blue", "tab:cyan", "tab:green", "tab:red"]))
plt.figure(figsize=(7, 6))
for g, ws in groups.items():
    for w in ws:
        x, y = P[stoi[w]]; plt.scatter(x, y, c=colors[g], s=30); plt.annotate(w, (x, y), fontsize=8)
plt.legend(handles=[plt.Line2D([], [], marker="o", ls="", c=c, label=g) for g, c in colors.items()], fontsize=8)
plt.title("SGNS embeddings (PCA)"); plt.show()

## 6. Analogies - vector arithmetic
Relations appear as roughly constant offsets: `paris − france + italy ≈ rome`, `king − man + woman ≈ queen`.  
On a tiny synthetic corpus this is only approximate, so we print the top-3 candidates (input words excluded).

In [ ]:
def analogy(a, b, c, E=None, k=3):
    """a is to b as c is to ?  ->  b - a + c"""
    E = W if E is None else E
    target = F.normalize((E[stoi[b]] - E[stoi[a]] + E[stoi[c]]).unsqueeze(0), dim=-1)[0]
    sims = E @ target
    for w in (a, b, c): sims[stoi[w]] = -9
    return [itos[i] for i in sims.topk(k).indices.tolist()]

tests = [("france", "paris", "italy"), ("japan", "tokyo", "spain"), ("man", "king", "woman"),
         ("king", "queen", "uncle"), ("boy", "girl", "prince")]
for a, b, c in tests: print(f"{a} : {b} :: {c} : ?  ->", analogy(a, b, c))

def analogy_accuracy(E):
    cases = [(c1, p1, c2, p2) for (c1, p1), (c2, p2) in itertools.permutations(pairs_cc, 2)]
    return np.mean([analogy(c1, p1, c2, E=E, k=1)[0] == p2 for c1, p1, c2, p2 in cases])
print("country->capital analogy top-1 accuracy:", round(float(analogy_accuracy(W)), 3))

## 7. GloVe-style embeddings from co-occurrence counts
Instead of streaming (word, context) pairs, GloVe first builds a global **co-occurrence matrix** `X`, then fits vectors so that  
$w_i^\top \tilde w_j + b_i + \tilde b_j \approx \log X_{ij}$, weighting each entry by $f(X_{ij})=\min\big((X_{ij}/x_{max})^{\alpha},1\big)$.

In [ ]:
def cooccurrence(tokens, window=3):
    X = torch.zeros(V, V)
    for sent in tokens:
        ids = [stoi[w] for w in sent]
        for i, c in enumerate(ids):
            for j in range(max(0, i - window), min(len(ids), i + window + 1)):
                if j != i: X[c, ids[j]] += 1.0 / abs(i - j)          # closer words count more
    return X
X = cooccurrence(tokens[:6000])
ii, jj = X.nonzero(as_tuple=True); xij = X[ii, jj]
print("non-zero co-occurrence entries:", len(xij), "of", V * V)

class GloVe(nn.Module):
    def __init__(self, V, d):
        super().__init__()
        self.w, self.wt = nn.Embedding(V, d), nn.Embedding(V, d)
        self.b, self.bt = nn.Embedding(V, 1), nn.Embedding(V, 1)
        for e in (self.w, self.wt): nn.init.normal_(e.weight, 0, 0.1)
        for e in (self.b, self.bt): nn.init.zeros_(e.weight)
    def forward(self, i, j, x, xmax=50.0, alpha=0.75):
        pred = (self.w(i) * self.wt(j)).sum(-1) + self.b(i).squeeze(-1) + self.bt(j).squeeze(-1)
        weight = torch.clamp((x / xmax) ** alpha, max=1.0)
        return (weight * (pred - torch.log(x)) ** 2).mean()

seed_everything()
glove = GloVe(V, 32); gopt = torch.optim.Adam(glove.parameters(), lr=0.05)
for step in range(301):
    loss = glove(ii, jj, xij); gopt.zero_grad(); loss.backward(); gopt.step()
    if step % 100 == 0: print(f"step {step:3d} | loss {loss.item():.4f}")
Wg = F.normalize((glove.w.weight + glove.wt.weight).detach(), dim=-1)        # sum of both tables (as in the paper)
print("GloVe neighbours of 'cat':", neighbours("cat", Wg), "\nGloVe analogy accuracy:", round(float(analogy_accuracy(Wg)), 3))

## 8. Sub-word (FastText-style) embeddings - handling unseen words
Word-level tables cannot represent a word they never saw (`parises`, `catlike`). **FastText** represents a word as the *sum of its character n-gram vectors* (hashed into a fixed number of buckets), so related spellings share parameters and OOV words still get a vector.

In [ ]:
def char_ngrams(word, nmin=3, nmax=5):
    w = f"<{word}>"
    return [w[i:i + n] for n in range(nmin, nmax + 1) for i in range(len(w) - n + 1)]
print(char_ngrams("paris"))

BUCKETS = 2000
def ngram_ids(word):
    ids = [hash_str(g) % BUCKETS for g in char_ngrams(word)]
    return ids

def hash_str(s):                       # deterministic (Python's hash() is salted per process)
    h = 2166136261
    for ch in s: h = ((h ^ ord(ch)) * 16777619) & 0xFFFFFFFF
    return h

class SubwordSGNS(nn.Module):
    def __init__(self, V, d):
        super().__init__()
        self.ng = nn.EmbeddingBag(BUCKETS, d, mode="mean"); self.out_embed = nn.Embedding(V, d)
        nn.init.uniform_(self.ng.weight, -0.5 / d, 0.5 / d); nn.init.zeros_(self.out_embed.weight)
        # pre-compute n-gram ids for each vocab word (one bag per word)
        bags = [ngram_ids(w) for w in itos]
        self.register_buffer("flat", torch.tensor(list(itertools.chain(*bags))))
        self.register_buffer("offs", torch.tensor([0] + list(itertools.accumulate(len(b) for b in bags[:-1]))))
        self.register_buffer("lens", torch.tensor([len(b) for b in bags]))
    def word_vectors(self):                                    # (V, d) built from n-grams
        return self.ng(self.flat, self.offs)
    def vec_for(self, word):                                   # works for ANY string
        return self.ng(torch.tensor(ngram_ids(word)), torch.tensor([0]))[0]
    def forward(self, centre, context, negatives):
        v = self.word_vectors()[centre]
        pos = F.logsigmoid((v * self.out_embed(context)).sum(-1))
        neg = F.logsigmoid(-(self.out_embed(negatives) @ v.unsqueeze(-1)).squeeze(-1)).sum(-1)
        return -(pos + neg).mean()

seed_everything()
sub = SubwordSGNS(V, 32); sopt = torch.optim.Adam(sub.parameters(), lr=1e-2)
for ep in range(2):
    perm = torch.randperm(len(centres))[:60000]
    for i in range(0, len(perm), 1024):
        idx = perm[i:i + 1024]
        neg = torch.multinomial(noise, len(idx) * 5, replacement=True).view(len(idx), 5)
        loss = sub(centres[idx], contexts[idx], neg); sopt.zero_grad(); loss.backward(); sopt.step()
    print("epoch", ep, "loss", round(loss.item(), 4))

with torch.no_grad():
    Ws = F.normalize(sub.word_vectors(), dim=-1)
    for oov in ["parises", "catlike", "romeo"]:                # never appear in the corpus
        q = F.normalize(sub.vec_for(oov), dim=0)
        top = (Ws @ q).topk(3).indices.tolist()
        print(f"OOV {oov!r:10s} -> nearest known words: {[itos[i] for i in top]}")

## 9. Using embeddings downstream
Initialise a model's `nn.Embedding` with the learned vectors, then **freeze** them (fast, stable, great for tiny data) or **fine-tune** them. Task: classify a word's group, training only on some words and testing on the **held-out** ones. Frozen embeddings carry the semantic structure, so a linear layer generalises to unseen words.

In [ ]:
labels = {w: g for g, ws in groups.items() for w in ws}
words = list(labels); gnames = list(groups)
rnd = random.Random(1); rnd.shuffle(words)
split = int(0.6 * len(words)); tr_w, te_w = words[:split], words[split:]
tr_x = torch.tensor([stoi[w] for w in tr_w]); tr_y = torch.tensor([gnames.index(labels[w]) for w in tr_w])
te_x = torch.tensor([stoi[w] for w in te_w]); te_y = torch.tensor([gnames.index(labels[w]) for w in te_w])

class WordClassifier(nn.Module):
    def __init__(self, vectors, freeze, n_cls):
        super().__init__()
        self.emb = nn.Embedding.from_pretrained(vectors.clone(), freeze=freeze)
        self.fc = nn.Linear(vectors.size(1), n_cls)
    def forward(self, x): return self.fc(self.emb(x))

def run(vectors, freeze, steps=200):
    m = WordClassifier(vectors, freeze, len(gnames)); opt = torch.optim.Adam([p for p in m.parameters() if p.requires_grad], lr=0.05)
    for _ in range(steps):
        loss = F.cross_entropy(m(tr_x), tr_y); opt.zero_grad(); loss.backward(); opt.step()
    return (m(te_x).argmax(1) == te_y).float().mean().item()

rand = F.normalize(torch.randn_like(W), dim=-1)
print(f"test accuracy on {len(te_w)} held-out words")
print("  random frozen embeddings  :", round(run(rand, True), 3))
print("  SGNS frozen embeddings    :", round(run(W, True), 3))
print("  GloVe frozen embeddings   :", round(run(Wg, True), 3))
print("  SGNS fine-tuned           :", round(run(W, False), 3))

## 10. Embeddings are not only for words
Any categorical feature (user id, product id, tool name, token position, time bucket) can get an embedding. Two patterns you will meet constantly:

In [ ]:
# (a) Entity embeddings for tabular / recommender models
class MatrixFactorisation(nn.Module):
    def __init__(self, n_users, n_items, d=8):
        super().__init__()
        self.u, self.i = nn.Embedding(n_users, d), nn.Embedding(n_items, d)
        self.ub, self.ib = nn.Embedding(n_users, 1), nn.Embedding(n_items, 1)
    def forward(self, u, i): return (self.u(u) * self.i(i)).sum(-1) + self.ub(u).squeeze(-1) + self.ib(i).squeeze(-1)

torch.manual_seed(0)
U, I = 40, 30
true_u, true_i = torch.randn(U, 3), torch.randn(I, 3)
uu, ii_ = torch.randint(0, U, (4000,)), torch.randint(0, I, (4000,))
ratings = (true_u[uu] * true_i[ii_]).sum(-1) + 0.1 * torch.randn(4000)
mf = MatrixFactorisation(U, I); mopt = torch.optim.Adam(mf.parameters(), lr=0.05, weight_decay=1e-4)
for step in range(200):
    loss = F.mse_loss(mf(uu[:3000], ii_[:3000]), ratings[:3000]); mopt.zero_grad(); loss.backward(); mopt.step()
print("train MSE:", round(loss.item(), 3), "| held-out MSE:", round(F.mse_loss(mf(uu[3000:], ii_[3000:]), ratings[3000:]).item(), 3))

# (b) Positional embeddings: a learned table indexed by position, added to token embeddings
tok_emb, pos_emb = nn.Embedding(100, 16), nn.Embedding(32, 16)
ids = torch.randint(0, 100, (2, 10))
x = tok_emb(ids) + pos_emb(torch.arange(10))
print("token + position embedding:", x.shape)

## 11. Saving, loading and exporting
- `torch.save(tensor)` for PyTorch pipelines.
- TSV files (`vectors.tsv` + `metadata.tsv`) can be uploaded to the TensorFlow Embedding Projector for interactive 3-D exploration.

In [ ]:
out = tempfile.gettempdir()
torch.save({"vectors": W, "itos": itos}, os.path.join(out, "word_vectors.pt"))
ckpt = torch.load(os.path.join(out, "word_vectors.pt"), weights_only=True)
print("reloaded:", ckpt["vectors"].shape, ckpt["itos"][:5])

with open(os.path.join(out, "vectors.tsv"), "w") as fv, open(os.path.join(out, "metadata.tsv"), "w") as fm:
    for w, v in zip(itos, W.tolist()):
        fv.write("\t".join(f"{x:.5f}" for x in v) + "\n"); fm.write(w + "\n")
print("wrote projector files to", out)

## 12. Key takeaways
| Concept | Remember |
|---|---|
| `nn.Embedding` | trainable lookup table, equivalent to `one_hot @ W` |
| Word2Vec / GloVe | *static* embeddings: one vector per word, learned from co-occurrence |
| Sub-word embeddings | handle rare / unseen words via character n-grams |
| Cosine similarity | compare **directions**; normalise vectors first |
| Limitation | static vectors cannot separate senses (*bank* of a river vs money) → contextual embeddings from Transformers (next notebook) |

## Exercises
1. Train SGNS with different `window`, `d` and `K`; plot analogy accuracy against each.
2. Implement **CBOW** (average the context vectors to predict the centre word) and compare with skip-gram.
3. Add **sub-sampling of frequent words** ($P(\text{keep}) = \sqrt{t/f}$) and measure the effect on training time and quality.
4. Replace `hash_str` bucketing with a learned vocabulary of the most frequent n-grams.
5. Load real GloVe vectors into `nn.Embedding.from_pretrained` and repeat the downstream classification.

---
### Summary & next step
Work through the exercises above before moving on. Next notebook: **08_sentence_embeddings_and_semantic_search.ipynb**.

*Notebook by **Chandan C** · [github.com/chandanc5525](https://github.com/chandanc5525) · Agentic AI BootCamp*